In [22]:
## 2. Ridership table

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("NYC Subway - Part 1 - explore").getOrCreate()
spark.version

'3.5.0'

In [2]:
rides = (spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv("/home/jovyan/project/data/raw/ridership_2024-07.csv")
)

In [3]:
rides.printSchema()

root
 |-- transit_timestamp: timestamp (nullable = true)
 |-- transit_mode: string (nullable = true)
 |-- station_complex_id: string (nullable = true)
 |-- station_complex: string (nullable = true)
 |-- borough: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- fare_class_category: string (nullable = true)
 |-- ridership: double (nullable = true)
 |-- transfers: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- georeference: string (nullable = true)



In [4]:
rides.show(5)

+-------------------+------------+------------------+--------------------+--------+--------------+--------------------+---------+---------+---------+---------+--------------------+
|  transit_timestamp|transit_mode|station_complex_id|     station_complex| borough|payment_method| fare_class_category|ridership|transfers| latitude|longitude|        georeference|
+-------------------+------------+------------------+--------------------+--------+--------------+--------------------+---------+---------+---------+---------+--------------------+
|2024-07-01 00:00:00|      subway|               336|       Hoyt St (2,3)|Brooklyn|     metrocard|Metrocard - Full ...|      3.0|      0.0|40.690544|-73.98506|POINT (-73.98506 ...|
|2024-07-01 00:00:00|      subway|               271|   Steinway St (M,R)|  Queens|     metrocard|Metrocard - Senio...|      1.0|      0.0|40.756878|-73.92074|POINT (-73.92074 ...|
|2024-07-01 00:00:00|      subway|               180|Kingston-Throop A...|Brooklyn|     metroca

In [5]:
rides.count()

2156907

In [6]:
rides.groupBy("transit_mode").count().show()

+--------------------+-------+
|        transit_mode|  count|
+--------------------+-------+
|                tram|   9186|
|staten_island_rai...|   8716|
|              subway|2139005|
+--------------------+-------+



In [7]:
rides.describe("ridership", "transfers", "latitude", "longitude").show()

+-------+------------------+------------------+------------------+-------------------+
|summary|         ridership|         transfers|          latitude|          longitude|
+-------+------------------+------------------+------------------+-------------------+
|  count|           2156907|           2156907|           2156907|            2156907|
|   mean| 44.30188135139809|1.8096148790837991| 40.73175981769349| -73.93535097191214|
| stddev|178.85408639858005|11.675011879301625|0.0780689013079846|0.05652950336380422|
|    min|               1.0|               0.0|         40.576126|          -74.07484|
|    max|           14387.0|            1288.0|         40.903126|           -73.7554|
+-------+------------------+------------------+------------------+-------------------+



In [8]:
rides.orderBy(rides.ridership.desc()).show(5)

+-------------------+------------+------------------+--------------------+---------+--------------+-------------------+---------+---------+---------+----------+--------------------+
|  transit_timestamp|transit_mode|station_complex_id|     station_complex|  borough|payment_method|fare_class_category|ridership|transfers| latitude| longitude|        georeference|
+-------------------+------------+------------------+--------------------+---------+--------------+-------------------+---------+---------+---------+----------+--------------------+
|2024-07-23 17:00:00|      subway|               610|Grand Central-42 ...|Manhattan|          omny|   OMNY - Full Fare|  14387.0|     32.0|40.751778|-73.976845|POINT (-73.976845...|
|2024-07-09 17:00:00|      subway|               610|Grand Central-42 ...|Manhattan|          omny|   OMNY - Full Fare|  14237.0|     50.0|40.751778|-73.976845|POINT (-73.976845...|
|2024-07-16 17:00:00|      subway|               610|Grand Central-42 ...|Manhattan|      

In [9]:
rides.where("ridership != floor(ridership)").count()

0

In [10]:
rides.selectExpr("min(transit_timestamp)", "max(transit_timestamp)").show()

+----------------------+----------------------+
|min(transit_timestamp)|max(transit_timestamp)|
+----------------------+----------------------+
|   2024-07-01 00:00:00|   2024-07-31 23:00:00|
+----------------------+----------------------+



In [11]:
rides.distinct().count()

2156907

In [12]:
rides.where("cast(station_complex_id as int) is null").groupBy("station_complex_id", "transit_mode").count().show()

+------------------+------------+-----+
|station_complex_id|transit_mode|count|
+------------------+------------+-----+
|             TRAM1|        tram| 4660|
|             TRAM2|        tram| 4526|
+------------------+------------+-----+



In [13]:
rides.groupBy("borough").count().show()

+-------------+------+
|      borough| count|
+-------------+------+
|       Queens|388549|
|     Brooklyn|755034|
|Staten Island|  8716|
|    Manhattan|671337|
|        Bronx|333271|
+-------------+------+



In [14]:
rides.where("transit_mode = 'subway' and cast(station_complex_id as int) is null").count()

0

In [21]:
## 2. Stations table

stations = (spark.read
            .option("header", True)
            .option("inferSchema", True)
            .csv("/home/jovyan/project/data/raw/stations.csv")
)

In [17]:
stations.printSchema()

root
 |-- GTFS Stop ID: string (nullable = true)
 |-- Station ID: integer (nullable = true)
 |-- Complex ID: integer (nullable = true)
 |-- Division: string (nullable = true)
 |-- Line: string (nullable = true)
 |-- Stop Name: string (nullable = true)
 |-- Borough: string (nullable = true)
 |-- CBD: boolean (nullable = true)
 |-- Daytime Routes: string (nullable = true)
 |-- Structure: string (nullable = true)
 |-- GTFS Latitude: double (nullable = true)
 |-- GTFS Longitude: double (nullable = true)
 |-- North Direction Label: string (nullable = true)
 |-- South Direction Label: string (nullable = true)
 |-- ADA: integer (nullable = true)
 |-- ADA Northbound: integer (nullable = true)
 |-- ADA Southbound: integer (nullable = true)
 |-- ADA Notes: string (nullable = true)
 |-- Georeference: string (nullable = true)



In [18]:
stations.show(5)

+------------+----------+----------+--------+----------------+--------------+-------+-----+--------------+---------+-------------+--------------+---------------------+---------------------+---+--------------+--------------+---------+--------------------+
|GTFS Stop ID|Station ID|Complex ID|Division|            Line|     Stop Name|Borough|  CBD|Daytime Routes|Structure|GTFS Latitude|GTFS Longitude|North Direction Label|South Direction Label|ADA|ADA Northbound|ADA Southbound|ADA Notes|        Georeference|
+------------+----------+----------+--------+----------------+--------------+-------+-----+--------------+---------+-------------+--------------+---------------------+---------------------+---+--------------+--------------+---------+--------------------+
|         R43|        37|        37|     BMT|          4th Av|         77 St|     Bk|false|             R|   Subway|    40.629742|     -74.02551|            Manhattan|            Bay Ridge|  0|             0|             0|      NaN|PO

In [19]:
stations.count()

496

In [20]:
stations.groupBy("Borough").count().show()

+-------+-----+
|Borough|count|
+-------+-----+
|     Bx|   70|
|      Q|   83|
|     Bk|  169|
|      M|  153|
|     SI|   21|
+-------+-----+



In [23]:
stations.select("Complex ID").distinct().count()

445

In [24]:
stations.groupBy("Complex ID").count().orderBy("count", ascending=False).show(5)

+----------+-----+
|Complex ID|count|
+----------+-----+
|       611|    5|
|       623|    4|
|       628|    4|
|       624|    4|
|       602|    3|
+----------+-----+
only showing top 5 rows



In [25]:
stations.groupBy("ADA Notes").count().orderBy("count", ascending=False).show(5)

+--------------------+-----+
|           ADA Notes|count|
+--------------------+-----+
|                 NaN|  487|
|       Downtown only|    3|
|         Uptown only|    2|
|Manhattan-bound only|    2|
|       Outbound only|    1|
+--------------------+-----+
only showing top 5 rows



In [28]:
## 3. Join check: ridership ↔ stations

ride_ids = rides.where("transit_mode = 'subway'").selectExpr("cast(station_complex_id as int) as complex_id").distinct()

In [29]:
station_ids = stations.selectExpr("`Complex ID` as complex_id").distinct()

In [30]:
ride_ids.join(station_ids, "complex_id", "left_anti").count()

0

In [31]:
station_ids.join(ride_ids, "complex_id", "left_anti").count()

21

In [34]:
# check for the 21 complexes without subway riderships
station_ids.join(ride_ids, "complex_id", "left_anti").join(stations.selectExpr("`Complex ID` as complex_id", "Borough"), "complex_id").groupBy("Borough").count().show()

+-------+-----+
|Borough|count|
+-------+-----+
|     SI|   21|
+-------+-----+



In [36]:
## 4. Weather


weather = (spark.read
           .option("header", True)
           .option("inferSchema", True)
           .csv("/home/jovyan/project/data/raw/weather_2024-07.csv")
)

In [37]:
weather.count()

965

In [38]:
len(weather.columns)

124

In [42]:
# We print the Columns
weather.columns

['STATION',
 'DATE',
 'REPORT_TYPE2',
 'SOURCE3',
 'BackupDirection',
 'BackupDistance',
 'BackupDistanceUnit',
 'BackupElements',
 'BackupElevation',
 'BackupElevationUnit',
 'BackupEquipment',
 'BackupLatitude',
 'BackupLongitude',
 'BackupName',
 'CoolingDegreeDaysSeasonToDate',
 'DailyAverageDewPointTemperature',
 'DailyAverageDryBulbTemperature',
 'DailyAverageRelativeHumidity',
 'DailyAverageSeaLevelPressure',
 'DailyAverageStationPressure',
 'DailyAverageWetBulbTemperature',
 'DailyAverageWindSpeed',
 'DailyCoolingDegreeDays',
 'DailyDepartureFromNormalAverageTemperature',
 'DailyHeatingDegreeDays',
 'DailyMaximumDryBulbTemperature',
 'DailyMinimumDryBulbTemperature',
 'DailyPeakWindDirection',
 'DailyPeakWindSpeed',
 'DailyPrecipitation',
 'DailySnowDepth',
 'DailySnowfall',
 'DailySustainedWindDirection',
 'DailySustainedWindSpeed',
 'DailyWeather',
 'HeatingDegreeDaysSeasonToDate',
 'HourlyAltimeterSetting',
 'HourlyDewPointTemperature',
 'HourlyDryBulbTemperature',
 'HourlyP

In [41]:
# we look at useful column only
weather.select("DATE", "REPORT_TYPE2", "HourlyDryBulbTemperature", "HourlyPrecipitation", "HourlyPresentWeatherType", "REM").show(10, truncate=False)

+-------------------+------------+------------------------+-------------------+------------------------+------------------------------------------------------------------------------------------------------------------------+
|DATE               |REPORT_TYPE2|HourlyDryBulbTemperature|HourlyPrecipitation|HourlyPresentWeatherType|REM                                                                                                                     |
+-------------------+------------+------------------------+-------------------+------------------------+------------------------------------------------------------------------------------------------------------------------+
|2024-07-01 00:00:00|SOD         |NULL                    |NULL               |NULL                    |NULL                                                                                                                    |
|2024-07-01 00:51:00|FM-15       |20.6                    |0.0                |NULL             

In [43]:
weather.groupBy("REPORT_TYPE2").count().show()

+------------+-----+
|REPORT_TYPE2|count|
+------------+-----+
|         SOM|    1|
|       FM-15|  744|
|         SOD|   31|
|       FM-16|  189|
+------------+-----+



In [44]:
# what values does the rain column contains to look if there are any values that are not numbers 
weather.groupBy("HourlyPrecipitation").count().orderBy("count", ascending=False).show(10)

+-------------------+-----+
|HourlyPrecipitation|count|
+-------------------+-----+
|                0.0|  683|
|               NULL|  152|
|                  T|   50|
|                0.2|    9|
|                1.0|    7|
|                0.5|    7|
|                0.3|    6|
|                0.7|    6|
|                1.2|    6|
|                0.8|    4|
+-------------------+-----+
only showing top 10 rows



In [45]:
# we check if the two report_type columns that are copies identical or not to see if we can keep only one 
weather.where("REPORT_TYPE2 != REPORT_TYPE95").count()

0

In [46]:
weather.where("NOT (SOURCE3 <=> SOURCE96)").count()

0